Final model training and evaluation with model configurations and decision threshold selected with use of the development set. The final model is trained on the development set (n = 450) and evaluated once on the test set (n = 50).

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments 
from transformers import set_seed

from sklearn.metrics import(
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    multilabel_confusion_matrix
)

In [ ]:
#predetermined seed for later randomisation
SEED = 42 

#define text column
text_col = "Text"

#define label column
label_columns = [
    "Meaninglessness",
    "Loneliness",
    "Death Anxiety",
    "Death Acceptance",
    "Identity Confusion",
    "Freedom Responsibility",
    "Engagement",
    "Solitude"
]

In [ ]:
dev_df = pd.read_csv("Development_set.csv", encoding="utf-8-sig")
test_df = pd.read_csv("Test_set.csv", encoding="utf-8-sig")

print("Development set shape:", dev_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
#check the column names to avoid spelling mistakes / whitespaces
print("Development set columns: ")
print(dev_df.columns.tolist())

print("Test set columns: ")
print(test_df.columns.tolist())

In [ ]:
# Rather code it so that it checks for whether all posts are either coded 0 or 1 and otherwise giving an error
def clean_label_value(value):

    if pd.isna(value):
        raise ValueError("Label value is NaN")
    
    if isinstance(value, str):
        value = value.strip()

    try:
        numeric_value = float(value)

    except Exception as err:
        raise ValueError(f"Unexpected label value: {value!r}") from err

        
    if numeric_value == 0:
        return 0
        
    elif numeric_value == 1:
        return 1
        
    else:
        raise ValueError (f"Label value must be 0 or 1, but found {value!r}")
        

In [ ]:
# Validate the label values and construct multilabel vectors
datasets = {
    "development": dev_df,
    "test": test_df,
}

required_columns = ["Post_id", text_col] + label_columns

for name, df in datasets.items():
    # Check if all required columns are present
    missing_columns = set(required_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns in {name} set: {missing_columns}")

    if df["Post_id"].isna().any():
        raise ValueError(f"Missing Post_id values in {name} set.")

    invalid_text_mask = ~df[text_col].map(lambda value: isinstance(value, str) and bool(value.strip()))
    if invalid_text_mask.any():
        print(df.loc[invalid_text_mask, ["Post_id", text_col]])
        raise ValueError(f"Invalid text values found in column '{text_col}' of {name} set.")

    # Label validation and cleaning
    for col in label_columns:
        df[col] = df[col].apply(clean_label_value)

        invalid_mask = ~df[col].isin([0, 1])
        if invalid_mask.any():
            print(df.loc[invalid_mask, ["Post_id", col]])
            raise ValueError(f"Invalid label values found in column '{col}' of {name} set.")
        
    df["labels"] = df[label_columns].astype(int).values.tolist()

    print(f"{name.capitalize()} set label validation completed.")


In [ ]:
# Check number of occurrences per label in the development set

print("\nDevelopment set:")

# Count appearances of each label
label_counts = dev_df[label_columns].sum()
print("Label counts: ", label_counts)

# Count posts with at least one label
labeled_posts = (dev_df[label_columns].sum(axis=1) > 0).sum()
print("Posts that contain at least one label: ", labeled_posts)

# Count posts with no label
posts_without_label = len(dev_df) - labeled_posts
print("Posts without any label: ", posts_without_label)

# Count all positive labels
labeled_total = dev_df[label_columns].sum().sum()
print("Total number of positive labels: ", labeled_total)

In [ ]:
def test_setfit_model(model, dataset, dataset_name, threshold):
    texts = dataset[text_col].tolist()
    # Codes as provided by human coding
    y_true = dataset[label_columns].to_numpy(dtype=int)


    y_proba = model.predict_proba(texts)

    if torch.is_tensor(y_proba):
        y_proba = y_proba.detach().cpu().numpy()

    else:
        y_proba = np.asarray(y_proba)

    assert y_true.shape == (len(dataset), len(label_columns))

    # Protect y_proba against a label-order or output shape error
    assert y_proba.shape == y_true.shape
    assert y_proba.shape[1] == len(label_columns)

    if not np.isfinite(y_proba).all():
        raise ValueError("Predicted probabilities contain non-finite values (NaN or Inf)")

    if not ((y_proba >= 0) & (y_proba <= 1)).all():
        raise ValueError("Predicted probabilities are not in the range [0, 1]")
    
    # Converting per-label probabilities into binary predictions 
    # using the global threshold selected from pooled out-of-fold development predictions
    y_pred = (y_proba >= threshold).astype(int)
    results = {
        "dataset": dataset_name,
        "macro_f1": f1_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_f1": f1_score(y_true, y_pred, average = "micro", zero_division = 0),
        "macro_precision": precision_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_precision": precision_score(y_true, y_pred, average = "micro", zero_division = 0),
        "macro_recall": recall_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_recall": recall_score(y_true, y_pred, average = "micro", zero_division = 0)
    }

    return results, y_true, y_pred, y_proba

In [ ]:
# Train on the development set with the final selected model and settings

# Protect against overlapping Post_id values between development and test sets
assert len(dev_df) == 450
assert len(test_df) == 50
assert not dev_df["Post_id"].isin(test_df["Post_id"]).any(), "Development and test sets contain overlapping Post_id values."
assert not dev_df[text_col].isin(test_df[text_col]).any(), "Development and test sets contain overlapping text content."
assert dev_df["Post_id"].is_unique
assert test_df["Post_id"].is_unique

# Initialise final train dataset
final_train_dataset = Dataset.from_pandas(
    dev_df[[text_col, "labels"]].reset_index(drop = True)
)

# Protect against any unexpected changes in the number of rows in the final training dataset
assert final_train_dataset.num_rows == 450

# Creating a list to collect the test metrics
test_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
test_predictions = []

threshold = 0.42 # Global decision threshold selected based on the pooled out-of-fold development predictions

# Define the settings for the model training
args = TrainingArguments(
    output_dir = f"setfit_output/gte_oversampling_unweighted_seed42_global_threshold_final_model_training",
    batch_size = (4, 2),
    num_epochs = (1, 16),
    sampling_strategy = "oversampling",
    seed = SEED,
    show_progress_bar = True, 
)

num_classes = len(label_columns)

# Reinitialise the random seed for reproducibility
set_seed(SEED)

# Initialise the SetFit Model and the selected Sentence Transformer model: "Alibaba-NLP/gte-base-en-v1.5"
model = SetFitModel.from_pretrained(
    "Alibaba-NLP/gte-base-en-v1.5",
    multi_target_strategy="one-vs-rest", # multilabel differentiable head with one binary output per label
    use_differentiable_head=True,
    head_params={"out_features": num_classes},
    trust_remote_code=True,
)

 # Initialise maximum sequence length for the model to 1536 tokens
model.model_body.max_seq_length = 1536
print(model.model_body.max_seq_length)

print("Model loaded. Train it before using for inference.")

 # Initialise the Trainer
trainer = Trainer(
    model=model,
    args = args,
    train_dataset = final_train_dataset,
    column_mapping={
        "Text": "text",
        "labels": "label"
    }
)

# Starting the training of SetFit
trainer.train()

# Saving the trained SetFit model
model.save_pretrained("setfit_model/gte_oversampling_unweighted_seed42_global_threshold_final_model")

In [ ]:
# Evaluate test set with the final trained model and the selected global threshold

# Initialise the evaluation of the test dataset
test_results, y_true_test, y_pred_test, y_proba_test = test_setfit_model(
    model,
    test_df,
    "test", 
    threshold,
)

print(test_results)
print("Actual positive label assignments:", y_true_test.sum())
print("Predicted positive label assignments:", y_pred_test.sum())

# Append test results to a list
test_results = test_results.copy()
test_results["model"] = "Alibaba-NLP/gte-base-en-v1.5"
test_results["sampling_strategy"] = "oversampling"
test_results["embedding_batch_size"] = 4
test_results["head_batch_size"] = 2
test_results["embedding_epochs"] = 1
test_results["head_epochs"] = 16
test_results["seed"] = SEED
test_results["loss_weighting"] = "unweighted" 
test_results["threshold_strategy"] = "global"   
test_results["threshold"] = threshold
test_results["max_sequence_length"] = 1536
test_results["n_test_posts"] = len(test_df)


test_metrics.append(test_results)

# Storing post-level test predictions and probabilities in a list
for idx in range(len(test_df)):
    test_post = test_df.iloc[idx]

    test_post_metrics ={
        "model": "Alibaba-NLP/gte-base-en-v1.5",
        "sampling_strategy": "oversampling",
        "threshold_strategy": "global",
        "threshold": threshold,
        "loss_weighting": "unweighted",
        "post_id": test_post["Post_id"]
    }

    for j, label in enumerate(label_columns):
        test_post_metrics[f"true_{label}"] = y_true_test[idx][j]
        test_post_metrics[f"pred_{label}"] = y_pred_test[idx][j]
        test_post_metrics[f"prob_{label}"] = y_proba_test[idx][j]


    test_predictions.append(test_post_metrics)

# Labelwise classificaion report

label_report = classification_report(
    y_true_test,
    y_pred_test,
    target_names= label_columns,
    zero_division= 0,
    output_dict= True,    
)
    
print(label_report)

# Storing per-label metrics in a list
for l in label_columns:
    label_retrieval = label_report.get(l)

    label_entry ={
        "model": "Alibaba-NLP/gte-base-en-v1.5",
        "sampling_strategy": "oversampling",
        "threshold_strategy": "global",
        "threshold": threshold,
        "loss_weighting": "unweighted",
        "label": l,
        "precision": label_retrieval["precision"],
        "recall": label_retrieval["recall"],
        "f1_score": label_retrieval["f1-score"],
        "support": label_retrieval["support"]
    }

    label_metrics.append(label_entry)



# Convert lists into pd dataframes for csv file storage
test_metrics_df = pd.DataFrame(test_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
test_predictions_df = pd.DataFrame(test_predictions)

# Store metrics dataframes as csv files 
test_metrics_df.to_csv("Gte_final_test_metrics.csv", index = False) 
label_metrics_df.to_csv("Gte_final_test_per_label_metrics.csv", index = False)
test_predictions_df.to_csv("Gte_final_test_predictions_metrics.csv", index = False)                   


In [ ]:
# Multilabel confusion matrix
conf_matrix = multilabel_confusion_matrix(y_true_test, y_pred_test)

for label, matrix in zip(label_columns, conf_matrix):
    print(f"\n{label}")
    print(matrix)

In [ ]:
# Preprocessing for the visualisation of the multilabel confusion matrix

confusion_rows = []

for label, matrix in zip(label_columns, conf_matrix):
    tn, fp, fn, tp = matrix.ravel()

    confusion_rows.append({
        "Label": label,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

confusion_df = pd.DataFrame(confusion_rows)

print(confusion_df)

confusion_df.to_csv("Gte_final_test_multilabel_confusion_matrix.csv", index = False)

# Visualisation as a heatmap of the multilabel confusion matrix
plt.figure(figsize = (8, 6))

sns.heatmap(
    confusion_df.set_index("Label"),
    annot = True,
    fmt = "d",
    cmap = "Blues"
)

plt.title("Per-Label Confusion Matrix Counts on the Test Set")
plt.xlabel("Classification Outcome")
plt.ylabel("Label")
plt.tight_layout()

# Save the heatmap as a PNG file and a PDF file 
plt.savefig("Gte_final_test_multilabel_confusion_matrix_heatmap.png", dpi = 300, bbox_inches = "tight") 
plt.savefig("Gte_final_test_multilabel_confusion_matrix_heatmap.pdf", bbox_inches = "tight")
plt.show()

In [ ]:
# Check number of occurrences per label in the test set

print("\nTest set:")

# Count appearances of each label
label_counts = test_df[label_columns].sum()
print("Label counts: ", label_counts)

# Count posts with at least one label
labeled_posts = (test_df[label_columns].sum(axis=1) > 0).sum()
print("Posts that contain at least one label: ", labeled_posts)

# Count posts with no label
posts_without_label = len(test_df) - labeled_posts
print("Posts without any label: ", posts_without_label)

# Count all positive labels
labeled_total = test_df[label_columns].sum().sum()
print("Total number of positive labels: ", labeled_total)